In [125]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, cross_validate

In [126]:
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

In [127]:
df=pd.read_csv('data_realistic.csv')


In [128]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200020 entries, 0 to 200019
Data columns (total 18 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Patient ID                200020 non-null  int64  
 1   Heart Rate                200020 non-null  float64
 2   Respiratory Rate          170017 non-null  float64
 3   Timestamp                 200020 non-null  object 
 4   Body Temperature          190019 non-null  float64
 5   Oxygen Saturation         180018 non-null  float64
 6   Systolic Blood Pressure   184019 non-null  float64
 7   Diastolic Blood Pressure  184019 non-null  float64
 8   Age                       200020 non-null  int64  
 9   Gender                    200020 non-null  object 
 10  Weight (kg)               176018 non-null  float64
 11  Height (m)                184019 non-null  float64
 12  Derived_HRV               150015 non-null  float64
 13  Derived_Pulse_Pressure    169282 non-null  f

In [129]:
df.describe()


,Patient ID,Heart Rate,Respiratory Rate,Body Temperature,Oxygen Saturation,Systolic Blood Pressure,Diastolic Blood Pressure,Age,Weight (kg),Height (m),Derived_HRV,Derived_Pulse_Pressure,Derived_BMI,Derived_MAP,Mortality
count,200020.000000,200020.000000,170017.000000,190019.000000,180018.000000,184019.000000,184019.000000,200020.000000,176018.000000,184019.000000,150015.000000,169282.000000,161939.000000,169282.000000,200020.000000
mean,100010.500000,82.890301,15.481389,36.748000,96.954473,124.946184,79.504657,53.446275,75.579146,1.750216,0.100029,45.448093,25.194790,94.653138,0.080002
std,57740.944759,21.393447,2.501544,0.458397,5.488907,11.349006,6.496389,20.786802,16.390765,0.144875,0.028844,13.092759,6.971574,5.744578,0.271297
min,1.000000,52.723651,8.139498,35.508309,30.007618,95.443942,55.110074,18.000000,48.504491,1.464868,0.050000,4.772163,12.260487,73.610572,0.000000
25%,50005.750000,69.915663,13.475264,36.372791,96.195922,116.920688,74.457655,35.000000,62.446393,1.625091,0.075062,36.686446,20.146128,90.702474,0.000000
50%,100010.500000,80.314711,15.476708,36.747721,97.486449,124.523709,79.488639,53.000000,75.121285,1.750693,0.100085,45.025822,24.353611,94.531823,0.000000
75%,150015.250000,90.669993,17.486635,37.122621,98.758238,132.152646,84.573728,71.000000,87.800529,1.875431,0.124926,53.451364,29.332246,98.391580,0.000000
max,200020.000000,249.971321,22.870946,38.084075,100.000000,219.982332,102.482852,89.000000,223.042344,2.030446,0.149998,150.253671,95.773763,134.739235,1.000000


In [130]:
df.head()

,Patient ID,Heart Rate,Respiratory Rate,Timestamp,Body Temperature,Oxygen Saturation,Systolic Blood Pressure,Diastolic Blood Pressure,Age,Gender,Weight (kg),Height (m),Derived_HRV,Derived_Pulse_Pressure,Derived_BMI,Derived_MAP,Risk Category,Mortality
0,1,59.839814,11.906495,2024-01-01 01:00:00,36.787533,94.471191,127.719493,89.652503,37,Female,91.070942,NaN,NaN,38.066990,NaN,102.341500,Low Risk,0
1,2,62.516999,19.070790,2024-01-01 02:00:00,36.190589,96.922160,122.920439,86.713712,77,Male,51.815727,1.992067,0.117062,36.206728,13.057314,98.782621,Low Risk,0
2,3,62.801692,14.773531,2024-01-01 04:00:00,37.342493,100.000000,132.197070,NaN,68,Female,89.700391,1.755619,NaN,NaN,29.102726,NaN,Low Risk,0
3,4,102.209956,14.784302,2024-01-01 06:00:00,36.617834,94.306351,120.665649,NaN,41,Female,NaN,1.828423,0.064475,NaN,NaN,NaN,Low Risk,0
4,5,70.006988,16.164734,2024-01-01 07:00:00,36.891599,98.390368,137.669185,74.834344,25,Female,55.486607,1.865452,0.118484,62.834841,15.944837,95.779291,Low Risk,0


In [131]:
df.isnull().sum()

Patient ID                      0
Heart Rate                      0
Respiratory Rate            30003
Timestamp                       0
Body Temperature            10001
Oxygen Saturation           20002
Systolic Blood Pressure     16001
Diastolic Blood Pressure    16001
Age                             0
Gender                          0
Weight (kg)                 24002
Height (m)                  16001
Derived_HRV                 50005
Derived_Pulse_Pressure      30738
Derived_BMI                 38081
Derived_MAP                 30738
Risk Category                   0
Mortality                       0
dtype: int64

In [132]:
df.duplicated().sum()
df.drop_duplicates(inplace=True)

In [133]:
for col in df.select_dtypes(include='number'):
    df[col].fillna(df[col].median(),inplace=True)

/tmp/ipykernel_13792/1252447875.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(),inplace=True)
/tmp/ipykernel_13792/1252447875.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try u

In [134]:
df.isnull().sum()

Patient ID                  0
Heart Rate                  0
Respiratory Rate            0
Timestamp                   0
Body Temperature            0
Oxygen Saturation           0
Systolic Blood Pressure     0
Diastolic Blood Pressure    0
Age                         0
Gender                      0
Weight (kg)                 0
Height (m)                  0
Derived_HRV                 0
Derived_Pulse_Pressure      0
Derived_BMI                 0
Derived_MAP                 0
Risk Category               0
Mortality                   0
dtype: int64

In [135]:
df['Risk Category']=df['Risk Category'].map({
    'High Risk':1,
    'Low Risk': 0
})

In [136]:
df["Gender"] = df["Gender"].map({
    "Male": 1,
    "Female": 0
})


In [137]:
df.shape

(200020, 18)

In [138]:
X = df.drop(['Risk Category','Timestamp','Patient ID','Mortality'], axis=1)
y = df['Risk Category']

In [139]:
X.columns

Index(['Heart Rate', 'Respiratory Rate', 'Body Temperature',
       'Oxygen Saturation', 'Systolic Blood Pressure',
       'Diastolic Blood Pressure', 'Age', 'Gender', 'Weight (kg)',
       'Height (m)', 'Derived_HRV', 'Derived_Pulse_Pressure', 'Derived_BMI',
       'Derived_MAP'],
      dtype='object')

In [140]:
df.shape

(200020, 18)

In [141]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        class_weight="balanced",
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2


In [154]:
print(y_train.value_counts())
print(pipeline.named_steps['model'].classes_)


Risk Category
0    126038
1     33978
Name: count, dtype: int64
[0 1]


In [143]:
y_pred = pipeline.predict(X_test)  # Predicted classes
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

In [144]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

Accuracy: 0.6500 (65.00%)


In [145]:
precision = precision_score(y_test, y_pred, pos_label=1)
print(f"Precision: {precision:.4f}")

Precision: 0.3366


In [146]:
recall = recall_score(y_test, y_pred, pos_label=1)
print(f"Recall: {recall:.4f}")

Recall: 0.6676


In [147]:
f1 = f1_score(y_test, y_pred, pos_label=1)
print(f"F1-Score: {f1:.4f}")

F1-Score: 0.4476


In [148]:
FEATURE_ORDER = [
    "Heart Rate",
    "Respiratory Rate",
    "Body Temperature",
    "Oxygen Saturation",
    "Systolic Blood Pressure",
    "Diastolic Blood Pressure",
    "Age",
    "Gender",
    "Weight (kg)",
    "Height (m)",
    "Derived_HRV",
    "Derived_Pulse_Pressure",
    "Derived_BMI",
    "Derived_MAP"
]


In [182]:
patient_input = {
    "Heart Rate": 150,               # Tachycardia, >140 bpm can be critical
    "Respiratory Rate": 35,          # Severe tachypnea, >30 is alarming
    "Body Temperature": 40.5,        # Hyperpyrexia, >40°C is dangerous
    "Oxygen Saturation": 85,         # Hypoxemia, <90% is critical
    "Systolic Blood Pressure": 180,  # Hypertensive crisis
    "Diastolic Blood Pressure": 120, # Hypertensive crisis
    "Age": 65,                        # Older age increases risk
    "Gender": 1,                      # Gender encoding same as your system
    "Weight (kg)": 120,               # Obesity-related risk
    "Height (m)": 1.75,               # Same as normal
    "Derived_HRV": 20,                # Very low HRV indicates poor autonomic function
    "Derived_Pulse_Pressure": 60,     # High PP indicates arterial stiffness
    "Derived_BMI": 39,                # Obesity class II-III
    "Derived_MAP": 130                 # Mean arterial pressure dangerously high
}


In [168]:
import pandas as pd

patient_input_df = pd.DataFrame([patient_input])  # keep keys as column names
y_pred = pipeline.predict(patient_input_df)
y_pred_proba = pipeline.predict_proba(patient_input_df)[:, 1]  # risk probability
y_pred, y_pred_proba

(array([0]), array([0.4775769]))

In [165]:
input_array = np.array([[
    patient_input.get(col, np.nan)
    for col in FEATURE_ORDER
]])



In [183]:
prediction = pipeline.predict(patient_input_df)

print("HIGH RISK" if prediction[0] == 1 else "LOW RISK")


LOW RISK


In [162]:
list=X.columns.tolist()
list

['Heart Rate',
 'Respiratory Rate',
 'Body Temperature',
 'Oxygen Saturation',
 'Systolic Blood Pressure',
 'Diastolic Blood Pressure',
 'Age',
 'Gender',
 'Weight (kg)',
 'Height (m)',
 'Derived_HRV',
 'Derived_Pulse_Pressure',
 'Derived_BMI',
 'Derived_MAP']

In [163]:
for col in list:
    if col in FEATURE_ORDER:
        continue
    else:
        print(f"{col} is NOT in the model features.")

In [ ]:
patient_input_df = pd.DataFrame([patient_input])

patient_input_df = patient_input_df[X_train.columns]
prediction = pipeline.predict(patient_input_df)
print("HIGH RISK" if prediction[0] == 1 else "LOW RISK")
y_pred_proba = pipeline.predict_proba(patient_input_df)[:,1]
print("Risk Probability:", y_pred_proba[0])


LOW RISK
Risk Probability: 0.3784090140104203
